In [1]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torchvision

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

from PIL import Image

print("PyTorch Version:", torch.__version__)
print("TorchVision Version:", torchvision.__version__)

PyTorch Version: 2.13.0+cu126
TorchVision Version: 0.28.0+cu126


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [3]:
DATASET_PATH = r"D:\major project\tomato"

In [4]:
import os

print("Dataset Exists:", os.path.exists(DATASET_PATH))
print("Folders:\n")

for folder in os.listdir(DATASET_PATH):
    print(folder)

Dataset Exists: True
Folders:

Tomato___Bacterial_spot
Tomato___Early_blight
Tomato___healthy
Tomato___Late_blight
Tomato___Leaf_Mold
Tomato___Septoria_leaf_spot
Tomato___Spider_mites Two-spotted_spider_mite
Tomato___Target_Spot
Tomato___Tomato_mosaic_virus
Tomato___Tomato_Yellow_Leaf_Curl_Virus


In [5]:
import os
import shutil
from sklearn.model_selection import train_test_split

In [6]:
# Original dataset path
DATASET_PATH = r"D:\major project\tomato"

# Output directory
OUTPUT_PATH = r"D:\major project\dataset_split"

TRAIN_DIR = os.path.join(OUTPUT_PATH, "train")
VALID_DIR = os.path.join(OUTPUT_PATH, "valid")
TEST_DIR = os.path.join(OUTPUT_PATH, "test")

In [7]:
for folder in [TRAIN_DIR, VALID_DIR, TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("✅ Output folders created successfully!")

✅ Output folders created successfully!


In [8]:
# Split each class into Train (70%), Validation (15%), and Test (15%)

for class_name in os.listdir(DATASET_PATH):

    class_path = os.path.join(DATASET_PATH, class_name)

    # Skip if not a folder
    if not os.path.isdir(class_path):
        continue

    # Get all image files
    images = os.listdir(class_path)

    # First split: 70% Train, 30% Temp
    train_images, temp_images = train_test_split(
        images,
        test_size=0.30,
        random_state=42,
        shuffle=True
    )

    # Second split: 15% Validation, 15% Test
    valid_images, test_images = train_test_split(
        temp_images,
        test_size=0.50,
        random_state=42,
        shuffle=True
    )

    # Create class folders
    os.makedirs(os.path.join(TRAIN_DIR, class_name), exist_ok=True)
    os.makedirs(os.path.join(VALID_DIR, class_name), exist_ok=True)
    os.makedirs(os.path.join(TEST_DIR, class_name), exist_ok=True)

    # Copy training images
    for image in train_images:
        shutil.copy(
            os.path.join(class_path, image),
            os.path.join(TRAIN_DIR, class_name, image)
        )

    # Copy validation images
    for image in valid_images:
        shutil.copy(
            os.path.join(class_path, image),
            os.path.join(VALID_DIR, class_name, image)
        )

    # Copy test images
    for image in test_images:
        shutil.copy(
            os.path.join(class_path, image),
            os.path.join(TEST_DIR, class_name, image)
        )

    print(f"✅ {class_name} -> Train: {len(train_images)}, Validation: {len(valid_images)}, Test: {len(test_images)}")

print("\n🎉 Dataset splitting completed successfully!")

✅ Tomato___Bacterial_spot -> Train: 1557, Validation: 334, Test: 334
✅ Tomato___Early_blight -> Train: 751, Validation: 161, Test: 162
✅ Tomato___healthy -> Train: 1113, Validation: 239, Test: 239
✅ Tomato___Late_blight -> Train: 1407, Validation: 301, Test: 302
✅ Tomato___Leaf_Mold -> Train: 725, Validation: 156, Test: 156
✅ Tomato___Septoria_leaf_spot -> Train: 1335, Validation: 286, Test: 287
✅ Tomato___Spider_mites Two-spotted_spider_mite -> Train: 1173, Validation: 251, Test: 252
✅ Tomato___Target_Spot -> Train: 982, Validation: 211, Test: 211
✅ Tomato___Tomato_mosaic_virus -> Train: 291, Validation: 63, Test: 63
✅ Tomato___Tomato_Yellow_Leaf_Curl_Virus -> Train: 3798, Validation: 814, Test: 814

🎉 Dataset splitting completed successfully!


In [9]:
def count_images(folder_path):
    total = 0

    print(f"\n📂 {os.path.basename(folder_path).upper()}")

    for class_name in sorted(os.listdir(folder_path)):
        class_path = os.path.join(folder_path, class_name)

        if os.path.isdir(class_path):
            count = len(os.listdir(class_path))
            total += count
            print(f"{class_name:<40} : {count}")

    print("-" * 55)
    print(f"Total Images: {total}")


count_images(TRAIN_DIR)
count_images(VALID_DIR)
count_images(TEST_DIR)


📂 TRAIN
Tomato___Bacterial_spot                  : 1557
Tomato___Early_blight                    : 751
Tomato___Late_blight                     : 1407
Tomato___Leaf_Mold                       : 725
Tomato___Septoria_leaf_spot              : 1335
Tomato___Spider_mites Two-spotted_spider_mite : 1173
Tomato___Target_Spot                     : 982
Tomato___Tomato_Yellow_Leaf_Curl_Virus   : 3798
Tomato___Tomato_mosaic_virus             : 291
Tomato___healthy                         : 1113
-------------------------------------------------------
Total Images: 13132

📂 VALID
Tomato___Bacterial_spot                  : 334
Tomato___Early_blight                    : 161
Tomato___Late_blight                     : 301
Tomato___Leaf_Mold                       : 156
Tomato___Septoria_leaf_spot              : 286
Tomato___Spider_mites Two-spotted_spider_mite : 251
Tomato___Target_Spot                     : 211
Tomato___Tomato_Yellow_Leaf_Curl_Virus   : 814
Tomato___Tomato_mosaic_virus             : 6

In [10]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [11]:
IMAGE_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(20),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [12]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

valid_dataset = datasets.ImageFolder(
    VALID_DIR,
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transform
)

In [13]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

In [14]:
print("Train Images :", len(train_dataset))
print("Validation Images :", len(valid_dataset))
print("Test Images :", len(test_dataset))

print("\nClasses:")
print(train_dataset.classes)

Train Images : 13132
Validation Images : 2816
Test Images : 2820

Classes:
['Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']


In [15]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Load pretrained EfficientNetB0
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

print(model)

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          

In [16]:
import torch.nn as nn

NUM_CLASSES = 10

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=10, bias=True)
)


In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print("Device:", device)

Device: cuda


In [18]:
# Freeze all layers first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze classifier
for param in model.classifier.parameters():
    param.requires_grad = True

# Unfreeze the last EfficientNet feature block
for param in model.features[-1].parameters():
    param.requires_grad = True

In [19]:
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer (Only classifier parameters will be updated)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)


# Learning Rate Scheduler
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=3
)

In [20]:
NUM_EPOCHS = 30

best_val_loss = float("inf")
patience = 5
counter = 0

In [21]:
import os

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "best_efficientnetb0.pth")

print("Model will be saved to:", MODEL_PATH)

Model will be saved to: models\best_efficientnetb0.pth


In [22]:
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [23]:
def validate(model, dataloader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in tqdm(dataloader):

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [24]:
import os
import torch

# Create models folder
os.makedirs("models", exist_ok=True)

best_val_loss = float("inf")
best_val_acc = 0.0
counter = 0

for epoch in range(NUM_EPOCHS):

    print("=" * 60)
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")

    # -------------------- Train --------------------
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    # ------------------ Validation -----------------
    val_loss, val_acc = validate(
        model,
        valid_loader,
        criterion,
        device
    )

    # Update Learning Rate
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    # -------------------- Results ------------------
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.2f}%")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.2f}%")
    print(f"Learning Rate : {current_lr:.6f}")

    # ---------- Save checkpoint every epoch ----------
    torch.save(
        model.state_dict(),
        f"models/epoch_{epoch+1}.pth"
    )

    print(f"📁 Checkpoint Saved : epoch_{epoch+1}.pth")

    # ---------- Save Best Model ----------
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_val_acc = val_acc
        counter = 0

        torch.save(
            model.state_dict(),
            "models/best_efficientnetb0.pth"
        )

        print("✅ Best Model Updated")

    else:

        counter += 1
        print(f"Early Stopping Counter : {counter}/{patience}")

        if counter >= patience:
            print("🛑 Early Stopping Triggered")
            break

print("\n================ TRAINING COMPLETED ================")
print(f"Best Validation Accuracy : {best_val_acc:.2f}%")
print(f"Best Validation Loss     : {best_val_loss:.4f}")
print("Best Model Path          : models/best_efficientnetb0.pth")

Epoch [1/30]


100%|██████████| 88/88 [01:17<00:00,  1.13it/s]


Train Loss : 1.3237
Train Acc  : 65.06%
Val Loss   : 0.7200
Val Acc    : 82.85%
Learning Rate : 0.000100
📁 Checkpoint Saved : epoch_1.pth
✅ Best Model Updated
Epoch [2/30]


100%|██████████| 88/88 [00:23<00:00,  3.74it/s]


Train Loss : 0.6920
Train Acc  : 80.90%
Val Loss   : 0.4533
Val Acc    : 88.64%
Learning Rate : 0.000100
📁 Checkpoint Saved : epoch_2.pth
✅ Best Model Updated
Epoch [3/30]


...

Train Loss : 0.1583
Train Acc  : 94.81%
Val Loss   : 0.1316
Val Acc    : 95.49%
Learning Rate : 0.000100
📁 Checkpoint Saved : epoch_28.pth
✅ Best Model Updated
Epoch [29/30]


100%|██████████| 88/88 [00:20<00:00,  4.28it/s]


Train Loss : 0.1589
Train Acc  : 94.72%
Val Loss   : 0.1354
Val Acc    : 95.60%
Learning Rate : 0.000100
📁 Checkpoint Saved : epoch_29.pth
Early Stopping Counter : 1/5
Epoch [30/30]


100%|██████████| 88/88 [00:22<00:00,  3.90it/s]

Train Loss : 0.1557
Train Acc  : 94.91%
Val Loss   : 0.1321
Val Acc    : 95.60%
Learning Rate : 0.000100
📁 Checkpoint Saved : epoch_30.pth
Early Stopping Counter : 2/5

================ TRAINING COMPLETED ================
Best Validation Accuracy : 95.49%
Best Validation Loss     : 0.1316
Best Model Path          : models/best_efficientnetb0.pth


In [5]:
os.makedirs("models", exist_ok=True)

In [2]:
import os

print("Current Working Directory:")
print(os.getcwd())

print("\nModel Folder:")
print(os.path.abspath("models"))

print("\nDoes models folder exist?")
print(os.path.exists("models"))

Current Working Directory:
C:\Users\lenovo\vite-project

Model Folder:
C:\Users\lenovo\vite-project\models

Does models folder exist?
True


In [3]:
import os

MODEL_DIR = r"D:\major project\models"
BEST_MODEL_PATH = os.path.join(MODEL_DIR, "best_efficientnetb0.pth")

In [4]:
import os

print("Notebook is running from:")
print(os.getcwd())

print("\nFiles in this folder:")
print(os.listdir())

Notebook is running from:
C:\Users\lenovo\vite-project

Files in this folder:
['.gitignore', '.ipynb_checkpoints', 'eslint.config.js', 'index.html', 'models', 'package.json', 'public', 'README.md', 'src', 'Tomato_Disease_EfficientNetB0.ipynb', 'vite.config.js']
